In [ ]:
pip install folium, geopandas

# 1. Problem Domain and Data Preparation

<b>Description of Problem Domain:</b>

The problem domain revolves around analysing CO2 emissions in Rwanda. This is crucial for environmental management and policy-making, especially as countries work towards reducing their carbon footprints in line with global climate goals.

<b>Motivation</b>: 
Understanding CO2 emissions helps in assessing the impact of various sectors on climate change and in formulating strategies to mitigate these effects.

<b>Dataset</b>
The dataset comprises open-source emissions data from Sentinel-5P satellites, covering the years 2019 to 2021.

<b>Challenges Faced</b>
1. Missing Values: Often, datasets have incomplete information.
2. Outliers: Certain data points may not fit the expected range and can skew results.
3. Data Size: Handling large datasets can be computationally intensive.

In [ ]:
# importing required libaries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium

from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

#### Data loading and overview

In [ ]:
data = pd.read_csv('data/dataset.csv')
data.head()

In [ ]:
data.shape

Dataset description

Each data point, or row, in the dataset is characterized by four location-based indices (latitude, longitude, year, and week number), 70 descriptive features, and a target variable representing emissions.

Feature Groups:
The 70 features are organized into eight distinct groups, each with a varying number of elements.

1. Sulfur Dioxide
2. Carbon Monoxide
3. Nitrogen Dioxide
4. Formaldehyde
5. UV Aerosol Index 
6. Ozone
7. Cloud
8. UvAerosolLayerHeight

more information about the dataset description can be found <a href='https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_SO2?hl=en#bands'>here</a>

The dataset includes information on SO? concentrations, cloud fractions, and viewing angles, processed using the Differential Optical Absorption Spectroscopy (DOAS) technique.


| Name                             | Units   | Min   | Max    | Description |
|----------------------------------|---------|-------|--------|-------------|
| SO2_column_number_density    | mol/m?  | -48*  | 0.24*  | SO? vertical column density at ground level, derived using the DOAS method. |
| SO2_column_number_density_amf | mol/m? | 0.1*  | 3.397* | Average air mass factor (AMF) weighted by cloud intensity. |
| SO2_slant_column_number_density | mol/m? | -0.147* | 0.162* | SO? slant column density, corrected for the ring effect. |
| cloud_fraction               | Fraction | 0*    | 1*     | Effective cloud fraction. For more details, refer to Sentinel 5P L2 Input/Output Data Definition Spec, p.220. |
| sensor_azimuth_angle         | degrees | -180* | 180*   | Azimuth angle of the satellite at the ground pixel (WGS84), measured East-of-North. |
| sensor_zenith_angle          | degrees | 0.09* | 67*    | Zenith angle of the satellite at the ground pixel (WGS84), measured away from the vertical. |
| solar_azimuth_angle          | degrees | -180* | 180*   | Azimuth angle of the Sun at the ground pixel (WGS84), measured East-of-North. |
| solar_zenith_angle           | degrees | 8*    | 80*    | Zenith angle of the Sun at the ground pixel (WGS84), measured away from the vertical. |
| SO2_column_number_density_15km | mol/m? | N/A   | N/A    | SO? vertical column density at 15 km altitude, derived using the DOAS method. |

### Data cleaning


1. Missing values:

UvAerosolLayerHeigh columns contain a high ratio of missing data points, so they would be dropped off.

2. Categorical features:

'week no' and 'year' would be classified as categorical data.

In [ ]:
UvAerosolLayerHeigh_columns = []
for column in data.columns:
    if 'UvAerosolLayerHeigh' in column:
        UvAerosolLayerHeigh_columns.append(column)


with pd.option_context("display.min_rows", 10):
    display(data[UvAerosolLayerHeigh_columns].isna().sum().sort_values())

In [ ]:
data.drop(columns=data[UvAerosolLayerHeigh_columns], axis=1, inplace=True)

In [ ]:
# Categorical data
categorical_data = ['year', 'week_no']
for column in categorical_data:
    data[column] = data[column].astype('category')

# 2. Exploratory Data Analysis (EDA)

1. Outliers: 

Sudden spikes in emissions data could be erroneous or due to unusual events.

In [ ]:
sns.boxplot(data.emission)
plt.title('CO2 emission outliers')  
plt.show()

In [ ]:
sns.lineplot(data = data, x = 'week_no', y = 'emission', errorbar = None)
plt.title('Emission Over Week')
plt.show()

In [ ]:
sns.lineplot(data = data, x = 'year', y = 'emission', errorbar = None)
plt.title('Emission Over Year')
plt.show()

2. Locations:

There are 497 locations in the dataset

In [ ]:
locations = data.groupby(['latitude', 'longitude']).size().sort_values()
locations.head()

In [ ]:
len(locations)

In [ ]:
plt.title("The 497 sensor devices")
plt.scatter(data.longitude, data.latitude, s=3)
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.show()

3. Geo Visualization

In [ ]:
geometry = gpd.points_from_xy(data.longitude, data.latitude)
geo_df = gpd.GeoDataFrame(
    data[["latitude", "longitude"]], geometry=geometry
)
geo_df.head()

In [ ]:
# run cell to view map
'''m = folium.Map(location=[data['latitude'].iloc[0], data['longitude'].iloc[0]], prefer_canvas=True)

for idx, row in data.iterrows():
    folium.CircleMarker(
        location=(row['latitude'], row['longitude']),
        radius=1,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(m)
'''

Correlation Analysis

There is not a strong correlation among these features in respect to the target (Emission)

In [ ]:
corr_features = abs(data.drop(columns=['ID_LAT_LON_YEAR_WEEK'], axis=1).corr()['emission']).sort_values(ascending = False).head(10)
corr_features

Categorical Analysis

In [ ]:
def cat_colum(column):
    grouped_data = data.groupby(column)['emission'].mean().sort_values(ascending=False).head(10)
    grouped_data.plot(kind='bar', color='skyblue')
    plt.title(f'Average CO2 Emissions by {column}')
    plt.xlabel('Region')
    plt.ylabel('Average CO2 Emissions')
    plt.xticks(rotation=45)
    plt.show()
    
for column in categorical_data:
    cat_colum(column)

<b>Feature Engineering</b>

Created three features from the data.

Month: derived from year and week number

Is_covid: derived from year ranging from March 2020 and January 2021

Is_lockdown: derived from year ranging from March-April 2024.

In [ ]:
data['month'] = data[['year', 'week_no']].apply(lambda row: datetime.datetime.strptime(f'{row["year"]}-{row["week_no"]+1}-1', "%Y-%W-%w").month, axis=1)
data['is_covid'] = (data['year'] == 2020) & (data['month'] > 2) | (data['year'] == 2021) & (data['month'] == 1)
data['is_covid'] = data['is_covid'].astype(np.int64)
data['is_lockdown'] = (data['year'] == 2020) & ((data['month'].isin([3,4]))) 
data['is_lockdown'] = data['is_lockdown'].astype(np.int64)
data.shape

# 3. Encoding Scheme & PCA for Dimensionality Reduction

Encoding categorical data

In [ ]:
X = data.drop(columns=['ID_LAT_LON_YEAR_WEEK', 'emission'], axis=1)
y = data['emission']

categorical_data = X.select_dtypes(include=['category']).columns.tolist()
onehot_encoder = OneHotEncoder(drop='first')
encoded_data = onehot_encoder.fit_transform(X[categorical_data])
X = X.drop(columns=categorical_data).reset_index(drop=True)
X = pd.concat([X, pd.DataFrame(encoded_data.toarray(), columns=onehot_encoder.get_feature_names_out(categorical_data))], axis=1)
X.shape

Dimensinality Reduction

In [ ]:
# fill missing values and scale the data
X = X.fillna(method='ffill').fillna(method='bfill')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# apply pca and calculate the cum variance
pca = PCA()
pca.fit(X_scaled)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
components = np.arange(1, len(cumulative_variance) + 1)

plt.plot(components, cumulative_variance, marker='x', linestyle='-', color='g', lw=2, label='Cumulative Variance')

plt.axhline(y=0.95, color='orange', linestyle='--', lw=2, label='95% Variance Threshold')
plt.text(len(cumulative_variance) * 0.2, 0.92, '95% cut-off', color='orange', fontsize=14)

plt.xticks(np.arange(0, len(cumulative_variance) + 1, step=10), fontsize=10, rotation=45)
plt.yticks(np.linspace(0, 1.1, num=12), fontsize=10)
plt.ylim(0, 1.05)

plt.xlabel('Number of Principal Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance', fontsize=12)
plt.title('Explained Variance by Number of Components (PCA)', fontsize=14, fontweight='bold')

plt.legend(loc='best', fontsize=12)
plt.grid(True, which='both', axis='both', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# Key Interpretation

The curve shows that the first few principal components capture a significant amount of the variance in the dataset. For example, the first 40?50 components account for roughly 70-80% of the variance. This indicates that most of the important patterns and variability in the data are captured by these early components.

The orange dashed line marks the point where 95% of the variance is explained. Based on the plot, it appears that around 75 components are required to explain 95% of the variance.
This means that out of the 122 features in the dataset, we can reduce the dimensionality to 75 components while retaining almost all of the original information (95% of it).